# 00 · Setup & API key check
Run each cell with **Shift + Enter**, top to bottom.

### 1. Install Packages

In [ ]:
   %pip install -q -U anthropic httpx python-dotenv brotli zstandard

### 2. Check that the Claude API key in `.env` works
This never prints the key itself.

In [ ]:
from dotenv import load_dotenv
import anthropic

load_dotenv(".env", override=True)
client = anthropic.Anthropic()

try:
    models = [m.id for m in client.models.list(limit=10)]
    print("SUCCESS - your key works.")
    print("Models you can use:", ", ".join(models[:6]))
except anthropic.AuthenticationError:
    print("The key was rejected - create a new one under Console -> API Keys and paste it into .env")

### 3. First Claude call: a coach note from your real latest day (that data provides)

In [ ]:
import pandas as pd

d = pd.read_csv("data/daily_scores.csv", index_col="date")
latest = d.dropna(subset=["readiness"]).iloc[-1]
cols = ["readiness","hrv_ms","hrv_baseline","resting_hr","rhr_baseline","sleep_hrs","impact_hrv","impact_rhr","impact_sleep","impact_resp"]
today = latest[cols].astype(float).round(1).to_dict()
print(latest.name, today)

msg = client.messages.create(
    model=models[0],   # newest model your key can use
    max_tokens=1000,
    system="You are a concise recovery coach. Use only the numbers given. 'impact_' fields are readiness points each signal added or removed.",
    messages=[{"role": "user", "content": f"Data for {latest.name}: {today}. Write a 3-sentence briefing with one recommendation."}],
)
print("\n" + "".join(b.text for b in msg.content if b.type == "text"))